# Install packages and import libraries

In [6]:
!pip -q install xgboost joblib

import os
import numpy as np
import pandas as pd
import joblib
import warnings

from sklearn.model_selection import LeaveOneGroupOut
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from xgboost import XGBClassifier

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

warnings.filterwarnings("ignore")

RANDOM_STATE = 42

print("Libraries imported successfully.")

Libraries imported successfully.


# Load the three feature datasets

In [8]:
# ============================================================
# Load revised WESAD feature datasets
# ============================================================

DATA_DIR = "/content"

df_bvp = pd.read_csv(
    os.path.join(DATA_DIR, "features_BVP_only_v2.csv")
)

df_acc = pd.read_csv(
    os.path.join(DATA_DIR, "features_ACC_only_v2.csv")
)

df_combined = pd.read_csv(
    os.path.join(DATA_DIR, "features_BVP_ACC_combined_v2.csv")
)

datasets = {
    "BVP_only": df_bvp,
    "ACC_only": df_acc,
    "BVP_ACC_combined": df_combined
}

print("Datasets loaded successfully.")

for name, df in datasets.items():
    print(f"{name}: {df.shape}")

Datasets loaded successfully.
BVP_only: (2733, 18)
ACC_only: (2733, 16)
BVP_ACC_combined: (2733, 27)


In [9]:
# ============================================================
# Check consistency across the three datasets
# ============================================================

metadata_cols = [
    "subject",
    "window_start_sec",
    "window_end_sec",
    "wesad_label",
    "target",
    "target_name",
    "label_purity"
]

reference = df_bvp[metadata_cols].reset_index(drop=True)

for name, df in datasets.items():

    missing = [
        col for col in metadata_cols
        if col not in df.columns
    ]

    if missing:
        raise ValueError(
            f"{name} is missing metadata columns: {missing}"
        )

    current = df[metadata_cols].reset_index(drop=True)

    if not reference.equals(current):
        raise ValueError(
            f"{name} does not have the same windows or labels as BVP-only."
        )

print("All three datasets have matching windows and labels.")

print("\nNumber of participants:",
      df_bvp["subject"].nunique())

print("\nParticipants:")
print(sorted(df_bvp["subject"].unique()))

print("\nLabel distribution:")
print(df_bvp["target_name"].value_counts())

print("\nTarget encoding:")
print(df_bvp[["wesad_label", "target", "target_name"]]
      .drop_duplicates()
      .sort_values("target"))

All three datasets have matching windows and labels.

Number of participants: 15

Participants:
['S10', 'S11', 'S13', 'S14', 'S15', 'S16', 'S17', 'S2', 'S3', 'S4', 'S5', 'S6', 'S7', 'S8', 'S9']

Label distribution:
target_name
baseline    1749
stress       984
Name: count, dtype: int64

Target encoding:
     wesad_label  target target_name
0              1       0    baseline
117            2       1      stress


In [10]:
# ============================================================
# Prepare feature matrices
# ============================================================

X_datasets = {}
feature_columns = {}

y = df_bvp["target"].astype(int).to_numpy()
groups = df_bvp["subject"].astype(str).to_numpy()

# Metadata must not be used as model input
metadata_cols = [
    "subject",
    "window_start_sec",
    "window_end_sec",
    "wesad_label",
    "target",
    "target_name",
    "label_purity"
]

for name, df in datasets.items():

    features = [
        col for col in df.columns
        if col not in metadata_cols
    ]

    X = df[features].copy()

    # Ensure all features are numeric
    X = X.apply(pd.to_numeric, errors="coerce")

    # Replace infinite values with missing values
    X = X.replace([np.inf, -np.inf], np.nan)

    X_datasets[name] = X
    feature_columns[name] = features

    print(f"\n{name}")
    print("Number of features:", len(features))
    print("Feature matrix:", X.shape)
    print("Missing values:", X.isna().sum().sum())

print("\nLabels:", y.shape)
print("Groups:", groups.shape)


BVP_only
Number of features: 11
Feature matrix: (2733, 11)
Missing values: 0

ACC_only
Number of features: 9
Feature matrix: (2733, 9)
Missing values: 0

BVP_ACC_combined
Number of features: 20
Feature matrix: (2733, 20)
Missing values: 0

Labels: (2733,)
Groups: (2733,)


# Classifier

In [11]:
# ============================================================
# Define model configurations
# ============================================================

def create_model(model_name):

    if model_name == "RF":

        model = RandomForestClassifier(
            n_estimators=300,
            random_state=RANDOM_STATE,
            class_weight="balanced",
            n_jobs=-1
        )

    elif model_name == "SVM":

        model = SVC(
            kernel="rbf",
            C=1.0,
            gamma="scale",
            class_weight="balanced"
        )

    elif model_name == "XGB":

        model = XGBClassifier(
            n_estimators=300,
            max_depth=3,
            learning_rate=0.03,
            subsample=0.8,
            colsample_bytree=0.8,
            reg_lambda=1.0,
            objective="binary:logistic",
            eval_metric="logloss",
            random_state=RANDOM_STATE,
            n_jobs=-1
        )

    else:
        raise ValueError(f"Unknown model: {model_name}")

    return model


# ============================================================
# Create preprocessing and model pipeline
# ============================================================

def create_pipeline(model_name):

    steps = [
        ("imputer", SimpleImputer(strategy="median"))
    ]

    # Standardization is important for SVM.
    # RF and XGBoost do not require feature scaling.
    if model_name == "SVM":
        steps.append(
            ("scaler", StandardScaler())
        )

    steps.append(
        ("classifier", create_model(model_name))
    )

    return Pipeline(steps)


model_names = ["RF", "SVM", "XGB"]

print("Models defined:")
print(model_names)

Models defined:
['RF', 'SVM', 'XGB']


In [12]:
# ============================================================
# Evaluation metrics
# ============================================================

def calculate_metrics(y_true, y_pred):

    tn, fp, fn, tp = confusion_matrix(
        y_true,
        y_pred,
        labels=[0, 1]
    ).ravel()

    specificity = (
        tn / (tn + fp)
        if (tn + fp) > 0 else 0
    )

    return {
        "Accuracy": accuracy_score(y_true, y_pred),
        "Balanced_Accuracy": balanced_accuracy_score(
            y_true, y_pred
        ),
        "Precision": precision_score(
            y_true, y_pred, zero_division=0
        ),
        "Recall": recall_score(
            y_true, y_pred, zero_division=0
        ),
        "F1": f1_score(
            y_true, y_pred, zero_division=0
        ),
        "Specificity": specificity
    }

# LOSO evaluation and final model training

This cell does two separate things:
- Evaluates each model by holding out one participant at a time.
- Trains a final pipeline on all WESAD participants for later external testing.
It also saves the feature column names with each model. This is important because your own data must have exactly the same feature columns.

In [13]:
# ============================================================
# LOSO cross-validation
# ============================================================

logo = LeaveOneGroupOut()

results = []
subject_results = []

oof_predictions = {}

for dataset_name, X in X_datasets.items():

    print("\n" + "=" * 65)
    print("FEATURE SET:", dataset_name)
    print("=" * 65)

    for model_name in model_names:

        experiment_name = f"{dataset_name}_{model_name}"

        print("\nTraining:", experiment_name)

        oof_pred = np.full(len(y), -1, dtype=int)

        # ----------------------------------------------------
        # Leave-One-Subject-Out cross-validation
        # ----------------------------------------------------

        for fold, (train_idx, test_idx) in enumerate(
            logo.split(X, y, groups),
            start=1
        ):

            X_train = X.iloc[train_idx]
            X_test = X.iloc[test_idx]

            y_train = y[train_idx]
            y_test = y[test_idx]

            held_out_subject = groups[test_idx][0]

            pipeline = create_pipeline(model_name)

            pipeline.fit(X_train, y_train)

            predictions = pipeline.predict(X_test)

            oof_pred[test_idx] = predictions

            # Participant-level evaluation
            metrics = calculate_metrics(
                y_test,
                predictions
            )

            subject_results.append({
                "Feature_Set": dataset_name,
                "Model": model_name,
                "Subject": held_out_subject,
                **metrics
            })

            print(
                f"Fold {fold:02d} | "
                f"Held-out: {held_out_subject} | "
                f"F1: {metrics['F1']:.3f}"
            )

        # ----------------------------------------------------
        # Pooled out-of-fold evaluation
        # ----------------------------------------------------

        if np.any(oof_pred == -1):
            raise RuntimeError(
                f"Some samples were not evaluated: {experiment_name}"
            )

        oof_predictions[experiment_name] = oof_pred

        overall_metrics = calculate_metrics(
            y,
            oof_pred
        )

        results.append({
            "Feature_Set": dataset_name,
            "Model": model_name,
            **overall_metrics
        })

        print("\nOverall LOSO results:")
        for metric, value in overall_metrics.items():
            print(f"{metric}: {value:.4f}")

        print("\nClassification report:")
        print(
            classification_report(
                y,
                oof_pred,
                target_names=["Baseline", "Stress"],
                zero_division=0
            )
        )

print("\nAll nine LOSO experiments completed.")


FEATURE SET: BVP_only

Training: BVP_only_RF
Fold 01 | Held-out: S10 | F1: 0.510
Fold 02 | Held-out: S11 | F1: 0.727
Fold 03 | Held-out: S13 | F1: 0.590
Fold 04 | Held-out: S14 | F1: 0.690
Fold 05 | Held-out: S15 | F1: 0.387
Fold 06 | Held-out: S16 | F1: 0.752
Fold 07 | Held-out: S17 | F1: 0.783
Fold 08 | Held-out: S2 | F1: 0.538
Fold 09 | Held-out: S3 | F1: 0.539
Fold 10 | Held-out: S4 | F1: 0.593
Fold 11 | Held-out: S5 | F1: 0.612
Fold 12 | Held-out: S6 | F1: 0.458
Fold 13 | Held-out: S7 | F1: 0.690
Fold 14 | Held-out: S8 | F1: 0.558
Fold 15 | Held-out: S9 | F1: 0.491

Overall LOSO results:
Accuracy: 0.7201
Balanced_Accuracy: 0.6891
Precision: 0.6192
Recall: 0.5783
F1: 0.5980
Specificity: 0.7999

Classification report:
              precision    recall  f1-score   support

    Baseline       0.77      0.80      0.79      1749
      Stress       0.62      0.58      0.60       984

    accuracy                           0.72      2733
   macro avg       0.70      0.69      0.69      2

In [14]:
# ============================================================
# Overall model comparison
# ============================================================

results_df = pd.DataFrame(results)

results_df = results_df.sort_values(
    by="F1",
    ascending=False
).reset_index(drop=True)

print("Overall LOSO model comparison:")
display(
    results_df.round(4)
)

# Save results
results_df.to_csv(
    "/content/WESAD_LOSO_model_comparison_v2.csv",
    index=False
)

print("\nSaved:")
print("/content/WESAD_LOSO_model_comparison_v2.csv")

Overall LOSO model comparison:


,Feature_Set,Model,Accuracy,Balanced_Accuracy,Precision,Recall,F1,Specificity
0,BVP_ACC_combined,SVM,0.7684,0.7670,0.6527,0.7622,0.7032,0.7719
1,BVP_only,SVM,0.7157,0.7410,0.5724,0.8313,0.6780,0.6507
2,BVP_ACC_combined,XGB,0.7709,0.7404,0.7025,0.6311,0.6649,0.8496
3,BVP_ACC_combined,RF,0.7772,0.7372,0.7358,0.5945,0.6577,0.8799
4,BVP_only,XGB,0.7303,0.7049,0.6285,0.6138,0.6211,0.7959
5,BVP_only,RF,0.7201,0.6891,0.6192,0.5783,0.5980,0.7999
6,ACC_only,SVM,0.7150,0.6842,0.6108,0.5742,0.5919,0.7942
7,ACC_only,RF,0.7237,0.6713,0.6584,0.4837,0.5577,0.8588
8,ACC_only,XGB,0.7245,0.6703,0.6634,0.4766,0.5547,0.8639



Saved:
/content/WESAD_LOSO_model_comparison_v2.csv


In [15]:
# ============================================================
# Participant-level performance
# ============================================================

subject_results_df = pd.DataFrame(subject_results)

print("Participant-level results:")
display(
    subject_results_df.round(4)
)

# Calculate mean and standard deviation across participants
subject_summary = (
    subject_results_df
    .groupby(["Feature_Set", "Model"])
    [["Accuracy", "Balanced_Accuracy",
      "Precision", "Recall", "F1", "Specificity"]]
    .agg(["mean", "std"])
)

print("\nMean and standard deviation across participants:")
display(subject_summary.round(4))

# Save participant results
subject_results_df.to_csv(
    "/content/WESAD_LOSO_participant_results_v2.csv",
    index=False
)

subject_summary.to_csv(
    "/content/WESAD_LOSO_participant_summary_v2.csv"
)

Participant-level results:


,Feature_Set,Model,Subject,Accuracy,Balanced_Accuracy,Precision,Recall,F1,Specificity
0,BVP_only,RF,S10,0.4681,0.5200,0.3910,0.7324,0.5098,0.3077
1,BVP_only,RF,S11,0.8207,0.7856,0.8148,0.6567,0.7273,0.9145
2,BVP_only,RF,S13,0.6503,0.6605,0.5111,0.6970,0.5897,0.6239
3,BVP_only,RF,S14,0.8098,0.7611,0.8478,0.5821,0.6903,0.9402
4,BVP_only,RF,S15,0.6054,0.5495,0.4510,0.3382,0.3866,0.7607
...,...,...,...,...,...,...,...,...,...
130,BVP_ACC_combined,XGB,S5,0.8846,0.8632,0.8621,0.7937,0.8264,0.9328
131,BVP_ACC_combined,XGB,S6,0.7308,0.6265,0.9444,0.2615,0.4096,0.9915
132,BVP_ACC_combined,XGB,S7,0.7901,0.8242,0.6344,0.9365,0.7564,0.7119
133,BVP_ACC_combined,XGB,S8,0.7418,0.6733,0.7568,0.4242,0.5437,0.9224



Mean and standard deviation across participants:


Accuracy         Balanced_Accuracy         Precision  \
                           mean     std              mean     std      mean   
Feature_Set      Model                                                        
ACC_only         RF      0.7234  0.0650            0.6704  0.0849    0.6920   
                 SVM     0.7146  0.1308            0.6830  0.1257    0.6760   
                 XGB     0.7241  0.0720            0.6690  0.0909    0.7280   
BVP_ACC_combined RF      0.7773  0.0896            0.7364  0.1039    0.7848   
                 SVM     0.7683  0.0995            0.7657  0.0979    0.6809   
                 XGB     0.7711  0.1108            0.7386  0.1164    0.7698   
BVP_only         RF      0.7203  0.0972            0.6881  0.0892    0.6597   
                 SVM     0.7160  0.1358            0.7408  0.1160    0.6045   
                 XGB     0.7306  0.1050            0.7039  0.0951    0.6685   

                                Recall              F1         Specificity  \
                           std    mean     std    mean     std        mean   
Feature_Set      Model                                                       
ACC_only         RF     0.1581  0.4820  0.2153  0.5372  0.1408      0.8587   
                 SVM    0.2104  0.5721  0.2110  0.5864  0.1575      0.7940   
                 XGB    0.1745  0.4740  0.2426  0.5269  0.1537      0.8640   
BVP_ACC_combined RF     0.1607  0.5927  0.2153  0.6418  0.1600      0.8801   
                 SVM    0.1425  0.7589  0.1696  0.7014  0.1147      0.7724   
                 XGB    0.1736  0.6272  0.2318  0.6500  0.1676      0.8500   
BVP_only         RF     0.1602  0.5759  0.1597  0.5945  0.1146      0.8002   
                 SVM    0.1587  0.8302  0.1099  0.6888  0.1176      0.6513   
                 XGB    0.1634  0.6115  0.1521  0.6209  0.1152      0.7963   

                                
                           std  
Feature_Set      Model          
ACC_only         RF     0.1132  
                 SVM    0.1989  
                 XGB    0.1424  
BVP_ACC_combined RF     0.1240  
                 SVM    0.1501  
                 XGB    0.1736  
BVP_only         RF     0.1670  
                 SVM    0.2048  
                 XGB    0.1727

In [16]:
# ============================================================
# Train final models using all WESAD participants
# ============================================================

final_models = {}

for dataset_name, X in X_datasets.items():

    print("\n" + "=" * 60)
    print("Final training:", dataset_name)
    print("=" * 60)

    for model_name in model_names:

        model_key = f"{dataset_name}_{model_name}"

        print("Fitting:", model_key)

        final_pipeline = create_pipeline(model_name)

        final_pipeline.fit(X, y)

        final_models[model_key] = {
            "pipeline": final_pipeline,
            "feature_columns": feature_columns[dataset_name],
            "feature_set": dataset_name,
            "model_name": model_name
        }

        print("Completed:", model_key)

print("\nTotal final models:", len(final_models))


Final training: BVP_only
Fitting: BVP_only_RF
Completed: BVP_only_RF
Fitting: BVP_only_SVM
Completed: BVP_only_SVM
Fitting: BVP_only_XGB
Completed: BVP_only_XGB

Final training: ACC_only
Fitting: ACC_only_RF
Completed: ACC_only_RF
Fitting: ACC_only_SVM
Completed: ACC_only_SVM
Fitting: ACC_only_XGB
Completed: ACC_only_XGB

Final training: BVP_ACC_combined
Fitting: BVP_ACC_combined_RF
Completed: BVP_ACC_combined_RF
Fitting: BVP_ACC_combined_SVM
Completed: BVP_ACC_combined_SVM
Fitting: BVP_ACC_combined_XGB
Completed: BVP_ACC_combined_XGB

Total final models: 9


In [17]:
# ============================================================
# Save final models and metadata
# ============================================================

MODEL_PATH = "/content/WESAD_9_trained_models_v2.joblib"

model_bundle = {
    "models": final_models,

    "model_names": model_names,

    "feature_sets": list(X_datasets.keys()),

    "label_mapping": {
        0: "baseline",
        1: "stress"
    },

    "sampling_rates": {
        "BVP": 64,
        "ACC": 32
    },

    "window_duration_sec": 10,

    "min_label_purity": 0.80,

    "ACC_WESAD_conversion": "raw_ACC / 64",

    "feature_version": "v2",

    "training_dataset": "WESAD",

    "number_of_subjects": len(np.unique(groups)),

    "number_of_training_windows": len(y)
}

joblib.dump(
    model_bundle,
    MODEL_PATH,
    compress=3
)

print("All nine models exported successfully.")
print("Model file:", MODEL_PATH)

All nine models exported successfully.
Model file: /content/WESAD_9_trained_models_v2.joblib


In [18]:
# ============================================================
# Verify exported models
# ============================================================

loaded_bundle = joblib.load(MODEL_PATH)

print("Export verification")
print("-" * 45)

print("Feature version:",
      loaded_bundle["feature_version"])

print("Number of subjects:",
      loaded_bundle["number_of_subjects"])

print("Number of training windows:",
      loaded_bundle["number_of_training_windows"])

print("\nExported models:")

for model_key, model_info in loaded_bundle["models"].items():

    print(
        f"{model_key}: "
        f"{len(model_info['feature_columns'])} features"
    )

print("\nTotal models:",
      len(loaded_bundle["models"]))

print("\nFile size:",
      round(os.path.getsize(MODEL_PATH) / (1024 ** 2), 2),
      "MB")

Export verification
---------------------------------------------
Feature version: v2
Number of subjects: 15
Number of training windows: 2733

Exported models:
BVP_only_RF: 11 features
BVP_only_SVM: 11 features
BVP_only_XGB: 11 features
ACC_only_RF: 9 features
ACC_only_SVM: 9 features
ACC_only_XGB: 9 features
BVP_ACC_combined_RF: 20 features
BVP_ACC_combined_SVM: 20 features
BVP_ACC_combined_XGB: 20 features

Total models: 9

File size: 13.89 MB


# Download the trained models

In [19]:
from google.colab import files

files.download(
    "/content/WESAD_9_trained_models_v2.joblib"
)

files.download(
    "/content/WESAD_LOSO_model_comparison_v2.csv"
)

files.download(
    "/content/WESAD_LOSO_participant_results_v2.csv"
)

files.download(
    "/content/WESAD_LOSO_participant_summary_v2.csv"
)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>